## Retrieve Colab evidence
Run after the main notebook completes. The compact transfer excludes large model tensors; the full run ZIP in `/content/artifacts/` retains them.

In [2]:
from pathlib import Path
import json, zipfile, io, base64
from IPython.display import display, HTML
run = Path(json.loads(Path('/content/artifacts/latest_run.json').read_text())['path'])
print('EXPORT RUN:', run.name)
# Transfer inspectable analytical evidence through notebook output; model weights remain in the full runtime ZIP.
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in run.rglob('*'):
        if path.is_file() and path.stat().st_size < 10_000_000:
            archive.write(path, str(path.relative_to(run)))
payload = base64.b64encode(buffer.getvalue()).decode()
print('Evidence ZIP bytes:', len(buffer.getvalue()))
display(HTML('<a download="'+run.name+'_evidence.zip" href="data:application/zip;base64,'+payload+'">Download analytical evidence (model weights are in the full runtime ZIP)</a>'))
print('Full model archive:', str(run)+'.zip')


EXPORT RUN: 20261102T152220909909Z_3d7d7c00a8
Evidence ZIP bytes: 3808845


Full model archive: /content/artifacts/20261102T152220909909Z_3d7d7c00a8.zip


In [3]:
# Split the full archive so Colab's contents endpoint never returns a multi-GB JSON response.
import hashlib
archive_path = Path(str(run)+'.zip')
assert zipfile.is_zipfile(archive_path), 'Wait for the main notebook to finish the ZIP.'
parts_dir = Path('/content/exports')/run.name
parts_dir.mkdir(parents=True, exist_ok=True)
whole_hash = hashlib.sha256()
parts = []
with archive_path.open('rb') as source:
    i = 0
    while True:
        block = source.read(32*1024*1024)
        if not block: break
        whole_hash.update(block)
        name = f'archive.part{i:04d}'
        (parts_dir/name).write_bytes(block)
        parts.append({'name':name, 'bytes':len(block), 'sha256':hashlib.sha256(block).hexdigest()})
        i += 1
(parts_dir/'parts.json').write_text(json.dumps({'archive_name':archive_path.name,
    'archive_bytes':archive_path.stat().st_size,'archive_sha256':whole_hash.hexdigest(),'parts':parts},indent=2))
print('PARTS FOLDER:', parts_dir)
print('ARCHIVE BYTES:', archive_path.stat().st_size, '| PARTS:',len(parts))


PARTS FOLDER: /content/exports/20261102T152220909909Z_3d7d7c00a8
ARCHIVE BYTES: 2537239893 | PARTS: 76
